# Week 18 Optional A: Hybrid Search

## BM25 + FAISS Dense + Reciprocal Rank Fusion

**Who is this for**: Data scientists who finished the Week 18 main notebook and want the
production-grade retrieval pattern that shows up in every serious RAG deployment.

## The Problem with Dense-Only Retrieval

Your Week 18 RAG pipeline uses dense embeddings (Titan V2) to find relevant policy chunks.
This works well for semantic queries like "what indicates suspicious activity?" but fails
on exact-term queries:

| Query | Dense retrieval | Why it fails |
|-------|-----------------|--------------|
| "31 CFR 1010.311" | Retrieves CTR-adjacent text | Regulation codes have no semantic neighbors |
| "$10,000 threshold" | Retrieves amount-adjacent text | Dollar amounts are ambiguous without context |
| "MCC 6051" | May miss the specific code | MCC codes are domain-specific tokens |

BM25 - the classic keyword-based retrieval algorithm - nails these exact-term queries
because it scores on term frequency, not embedding similarity.

## Hybrid Search: Best of Both Worlds

```
Query --> BM25 sparse (exact term match)  --\
      --> FAISS dense (semantic match)    --> RRF (k=60) --> Top 3 results
```

**Reciprocal Rank Fusion (RRF)** combines result lists by rank position, not raw scores.
For each result, its fused score is the sum of `1/(k+rank)` across all retrieval systems,
where k=60 is the value from Cormack et al. (SIGIR 2009). Rank positions are comparable
across systems; raw BM25 and cosine scores are not.

## Anthropic Contextual Retrieval

Prepending LLM-generated context to each chunk before indexing reduces retrieval failure
by 49% vs standard hybrid search (both BM25 and FAISS indexes get the prefix).
Source: Anthropic contextual retrieval blog, Nov 2024.

Section 3 implements this pattern.

## Prerequisites

- Week 18 main notebook completed (exercise or solution)
- No new installs - rank_bm25 is already installed via strands-agents-tools

# Section 0: Setup

Same SageMaker environment as Week 18 main. No new installs needed - `rank_bm25`
is a transitive dependency of `strands-agents-tools` and is already present in
the kernel from the main notebook session.

The fraud corpus and FAISS index from Week 18 main are reconstructed here so this
notebook is fully self-contained.

In [ ]:
# =============================================================================
# IMPORTS - same as Week 18 main, plus rank_bm25
# =============================================================================
# sagemaker==2.257.3 was pinned in the main notebook. If running this notebook
# in a fresh kernel, the version installed by the main notebook is still active.
# rank_bm25 is a transitive dep of strands-agents-tools - no install needed.

import os
import time
import boto3
import sagemaker
import pandas as pd
import numpy as np
from sagemaker import get_execution_role
from importlib.metadata import version as pkg_version

# rank_bm25: BM25Okapi is the standard Okapi BM25 variant
from rank_bm25 import BM25Okapi

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_aws import BedrockEmbeddings, ChatBedrockConverse
from langchain_core.documents import Document

from ragas import evaluate, EvaluationDataset, SingleTurnSample
from ragas.metrics import faithfulness, answer_relevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

for pkg in ["rank-bm25", "faiss-cpu", "ragas", "langchain", "numpy", "sagemaker"]:
    try:
        print(f"  {pkg:22s} {pkg_version(pkg)}")
    except Exception:
        print(f"  {pkg:22s} (version not found)")


# =============================================================================
# SAGEMAKER + BEDROCK SETUP
# =============================================================================
sess   = sagemaker.Session()
role   = get_execution_role()
AWS_REGION = sess.boto_region_name

os.environ["AWS_REGION"]         = AWS_REGION
os.environ["AWS_DEFAULT_REGION"] = AWS_REGION
os.environ["KNOWLEDGE_BASE_ID"]  = (
    os.environ.get("KNOWLEDGE_BASE_ID")
    or os.environ.get("STRANDS_KNOWLEDGE_BASE_ID")
    or "FARSQGTONR"
)
STRANDS_KNOWLEDGE_BASE_ID = os.environ["KNOWLEDGE_BASE_ID"]

MODEL_ID       = "us.anthropic.claude-3-haiku-20240307-v1:0"
EMBED_MODEL_ID = "amazon.titan-embed-text-v2:0"

langchain_llm         = ChatBedrockConverse(model=MODEL_ID, region_name=AWS_REGION)
bedrock_embeddings    = BedrockEmbeddings(model_id=EMBED_MODEL_ID, region_name=AWS_REGION)
bedrock_agent_runtime = boto3.client("bedrock-agent-runtime", region_name=AWS_REGION)

# RAGAS judge wrappers (same as Week 18 main)
evaluator_llm        = LangchainLLMWrapper(langchain_llm)
evaluator_embeddings = LangchainEmbeddingsWrapper(bedrock_embeddings)

print(f"\nAWS Region: {AWS_REGION}")
print(f"LLM:        {MODEL_ID}")
print(f"KB:         {STRANDS_KNOWLEDGE_BASE_ID}")

In [ ]:
# =============================================================================
# FRAUD CORPUS + FAISS DENSE INDEX (same corpus as Week 18 main)
# =============================================================================
FRAUD_POLICY_CORPUS = [
    ("ctr_rules.md",
     "Currency Transaction Report (CTR) rules under 31 CFR 1010.311. Banks "
     "must file a CTR for each transaction in currency of more than $10,000. "
     "Multiple transactions are aggregated when known to be conducted by or "
     "on behalf of the same person and result in cash in or cash out totaling "
     "more than $10,000 in any one business day. Structuring - breaking a "
     "transaction into smaller amounts to evade the CTR threshold - is itself "
     "a federal violation under 31 USC 5324."),
    ("ofac_screening.md",
     "OFAC screening requirements. All wire transfers must be screened against "
     "the OFAC Specially Designated Nationals (SDN) list before execution. "
     "International wire transfers involving countries on the OFAC sanctions "
     "list (including but not limited to Iran, North Korea, Syria, and Cuba) "
     "require additional review and may be blocked outright. False positives "
     "must be cleared within 24 hours."),
    ("wire_record_keeping.md",
     "Wire transfer recordkeeping under 31 CFR 1010.410. For any international "
     "wire transfer of $3,000 or more, the bank must retain the originator's "
     "name, address, account number, amount, execution date, payment "
     "instructions, beneficiary bank, and beneficiary name. Records must be "
     "retained for five years."),
    ("structuring_red_flags.md",
     "Structuring red flags. Multiple cash deposits of amounts just under "
     "$10,000 across consecutive days at the same or related accounts are a "
     "classic structuring pattern. Velocity anomalies - for example three or "
     "more transactions at unrelated merchants within 30 minutes - indicate "
     "potential card testing or account takeover."),
    ("account_takeover_patterns.md",
     "Account takeover (ATO) indicators. A password change followed within "
     "minutes by a wire transfer to a newly added payee from an unfamiliar IP "
     "is a high-confidence ATO signal. Transactions that originate from "
     "geographies inconsistent with the customer's historical footprint, "
     "especially from countries the customer has never transacted with, "
     "require hold and verification."),
    ("unusual_hours_rule.md",
     "Unusual hours rule. Transactions initiated between 1:00 AM and 5:00 AM "
     "local time that fall outside the customer's typical active window "
     "require enhanced monitoring. Two or more such transactions within a "
     "single session should trigger a SAR review."),
    ("new_payee_hold.md",
     "New payee large transfer hold. Wire transfers exceeding $5,000 to payees "
     "that were added to the account within the preceding 72 hours require "
     "two-factor customer verification and a 24-hour hold regardless of the "
     "customer's risk score."),
    ("high_risk_merchant_categories.md",
     "High-risk merchant category codes (MCCs). Cryptocurrency exchanges, "
     "offshore gambling platforms, and money transfer services are classified "
     "as high-risk MCCs. Transactions at these merchants for amounts over "
     "$1,000 require enhanced due diligence."),
]

# Chunk with 512/80 (same as Week 18 main index_b)
documents = [Document(page_content=text, metadata={"source": name})
             for name, text in FRAUD_POLICY_CORPUS]
splitter  = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=80)
chunks    = splitter.split_documents(documents)

# FAISS dense baseline index
faiss_index = FAISS.from_documents(chunks, bedrock_embeddings)

# Flat lists for BM25 (BM25Okapi needs plain Python lists)
CORPUS_TEXTS   = [c.page_content for c in chunks]
CORPUS_SOURCES = [c.metadata["source"] for c in chunks]

print(f"Corpus: {len(FRAUD_POLICY_CORPUS)} documents -> {len(chunks)} chunks (512/80)")
print(f"FAISS dense index built with Titan V2 embeddings.")

# Section 1: BM25 Sparse Retrieval

## Why Keyword Search Still Matters

Dense embeddings are great at capturing meaning, but they fail on queries that contain
exact tokens that carry the full information: regulation codes, dollar thresholds, MCC
codes, legal citations. In those cases, the token IS the concept -- there is no
"semantic neighbor" to find.

**BM25** (Best Match 25) is the classic probabilistic keyword retrieval algorithm used
by Elasticsearch, Lucene, and every serious search engine for decades. It scores a
document against a query using three signals:

- **Term Frequency (TF)**: How often does the query term appear in the document?
  More occurrences = higher relevance, but with diminishing returns (log scale).
- **Inverse Document Frequency (IDF)**: How rare is the term across the corpus?
  Common words ("the", "a") are downweighted. Rare terms ("structuring", "SDN") are
  upweighted.
- **Document Length Normalization**: Long documents are penalized so they don't
  dominate just by containing more words.

The BM25 score for term `t` in document `d` is:

```
score(d, t) = IDF(t) * (TF(t,d) * (k1+1)) / (TF(t,d) + k1*(1 - b + b*|d|/avgdl))
```

Where `k1=1.5` (TF saturation), `b=0.75` (length normalization). `rank_bm25`
implements `BM25Okapi`, the standard variant.

## The Failure Mode Comparison

| Query | Dense (Titan V2) result | BM25 result | Winner |
|-------|------------------------|-------------|--------|
| "31 CFR 1010.311" | CTR-adjacent text | Exact regulation chunk | BM25 |
| "suspicious activity report" | Semantically related policies | All docs with "SAR" | Tie |
| "MCC 6051" | High-risk merchant text | Exact MCC chunk | BM25 |
| "what should I do if I see unusual activity?" | Correct policy | May miss if phrased differently | Dense |

**The takeaway**: BM25 is precise on exact terms; dense is better on paraphrase and
conceptual queries. Hybrid search captures both.

## Building a BM25 Index with rank_bm25

```python
from rank_bm25 import BM25Okapi

# Tokenize: BM25 works on token lists, not raw strings
tokenized = [text.split() for text in corpus_texts]

# Build index (O(N*avg_doc_len))
bm25 = BM25Okapi(tokenized)

# Query: returns a score array aligned with corpus_texts
scores = bm25.get_scores("structuring cash deposits".split())
```

Tokenization is intentionally simple (`.split()`). BM25 works on exact token matches,
so we want whitespace tokenization -- NOT subword tokenizers like BPE. "31" and "CFR"
should be separate tokens; splitting on whitespace preserves them.

In [ ]:
# =============================================================================
# DEMO: BM25 index + side-by-side comparison with FAISS dense
# =============================================================================

# 1. Build the BM25 index
# Tokenize each chunk by splitting on whitespace.
# BM25 is a bag-of-words model -- exact token match is the whole point,
# so simple whitespace split is correct. Do NOT use subword tokenizers here.
tokenized_corpus = [text.split() for text in CORPUS_TEXTS]
bm25_index = BM25Okapi(tokenized_corpus)

print(f"BM25 index built over {len(tokenized_corpus)} chunks.")
print(f"Average document length: {np.mean([len(t) for t in tokenized_corpus]):.1f} tokens\n")

# 2. Two representative queries that expose each retriever's weakness
DEMO_QUERIES = [
    # Exact-term query -- BM25 should win
    "31 CFR 1010.311 currency transaction report",
    # Conceptual query -- dense should win
    "what happens if a customer wires money late at night",
]

def bm25_retrieve(query: str, k: int = 3):
    """Return top-k chunks by BM25 score."""
    scores = bm25_index.get_scores(query.split())
    top_k  = np.argsort(scores)[::-1][:k]
    return [(CORPUS_SOURCES[i], CORPUS_TEXTS[i][:120], round(float(scores[i]), 4))
            for i in top_k]

def dense_retrieve(query: str, k: int = 3):
    """Return top-k chunks by FAISS cosine similarity."""
    docs = faiss_index.similarity_search(query, k=k)
    return [(d.metadata["source"], d.page_content[:120], "cosine") for d in docs]

print("=" * 70)
for q in DEMO_QUERIES:
    print(f"\nQuery: {q!r}")
    print("\n  BM25 top-3:")
    for src, snippet, score in bm25_retrieve(q):
        print(f"    [{src}] score={score}  '{snippet}...'")
    print("\n  Dense top-3:")
    for src, snippet, score in dense_retrieve(q):
        print(f"    [{src}] sim={score}    '{snippet}...'")
    print("-" * 70)

## Lab 1: Build Your Own BM25 Retriever

**Goal**: Build a `bm25_retrieve_lab` function that accepts a query and `k`, and
returns the top-k chunks ranked by BM25 score. Then run a small hit-rate experiment:
for each test query, check whether the expected source document appears in the top-k
results for BM25 vs. dense.

**Steps**:

1. Tokenize `CORPUS_TEXTS` into a list of token lists using `.split()`.
   Store as `lab1_tokenized`.
2. Build a `BM25Okapi` index from `lab1_tokenized`. Store as `lab1_bm25`.
3. Implement `bm25_retrieve_lab(query, k=3)`:
   - Call `lab1_bm25.get_scores(query.split())`
   - Use `np.argsort` to get the top-k indices (descending)
   - Return a list of `(source, snippet_80chars, bm25_score)` tuples
4. Run the evaluation loop below (provided) that computes hit rates for both
   retrievers over `LAB1_QUERIES`.

**Expected output**: A printed table showing hit rate (%) for BM25 and dense at k=3.
The exact-term queries should show BM25 >= dense; the conceptual ones may vary.

```
Query                                     BM25-hit  Dense-hit
31 CFR 1010.311 currency threshold        True      False
high risk MCC cryptocurrency              True      True
what time does suspicious activity occur  False     True
```

In [ ]:
# Lab 1: BM25 retriever + hit-rate comparison
# -----------------------------------------------------------------------
# Step 1: tokenize the corpus for BM25
lab1_tokenized = None  # YOUR CODE

# Step 2: build the BM25Okapi index
lab1_bm25 = None  # YOUR CODE

# Step 3: implement the retrieval function
def bm25_retrieve_lab(query: str, k: int = 3):
    """Return top-k (source, snippet, score) tuples from BM25."""
    # YOUR CODE
    pass

# -----------------------------------------------------------------------
# Step 4: evaluation loop (do not modify below this line)
LAB1_QUERIES = [
    # (query_text, expected_source_file)
    ("31 CFR 1010.311 currency transaction",       "ctr_rules.md"),
    ("high risk MCC cryptocurrency exchange",      "high_risk_merchant_categories.md"),
    ("unusual hours 1 AM 5 AM transactions",       "unusual_hours_rule.md"),
    ("what indicates suspicious wire transfer",    "account_takeover_patterns.md"),
]

if bm25_retrieve_lab is not None and lab1_bm25 is not None:
    print(f"{'Query':<45} {'BM25-hit':>8}  {'Dense-hit':>9}")
    print("-" * 66)
    bm25_hits, dense_hits = 0, 0
    for q, expected_src in LAB1_QUERIES:
        bm25_results  = bm25_retrieve_lab(q, k=3)
        dense_results = dense_retrieve(q, k=3)
        bm25_hit  = any(src == expected_src for src, _, _ in bm25_results)
        dense_hit = any(src == expected_src for src, _, _ in dense_results)
        bm25_hits  += int(bm25_hit)
        dense_hits += int(dense_hit)
        print(f"  {q[:43]:<43} {str(bm25_hit):>8}  {str(dense_hit):>9}")
    n = len(LAB1_QUERIES)
    print(f"\nHit rate @ k=3:  BM25={bm25_hits}/{n}  Dense={dense_hits}/{n}")

# Section 2: Reciprocal Rank Fusion (RRF)

## The Score Incompatibility Problem

BM25 scores and cosine similarity scores live on completely different scales:

- BM25: unbounded positive float (e.g., 4.27, 0.83, 12.1) -- depends on corpus size and IDF
- Cosine similarity: [-1, 1], typically [0, 1] for non-negative embeddings

You cannot add them. A BM25 score of 4.0 is not "better" than a cosine score of 0.8;
they measure completely different things.

**Reciprocal Rank Fusion (RRF)** sidesteps this by using only rank positions:

```
RRF_score(doc) = sum over all retrievers: 1 / (k + rank_of_doc_in_that_retriever)
```

Where `k = 60` is the smoothing constant from Cormack et al. (SIGIR 2009). The k=60
value was empirically tuned to reduce the impact of highly-ranked documents while
still rewarding good rank positions. Rank positions are always comparable; raw scores
are not.

## How RRF Works: A Worked Example

Suppose BM25 ranks doc A at position 1 and doc B at position 5.
Dense ranks doc B at position 1 and doc A at position 10.

```
RRF(A) = 1/(60+1) + 1/(60+10) = 0.01639 + 0.01429 = 0.03068
RRF(B) = 1/(60+5) + 1/(60+1)  = 0.01538 + 0.01639 = 0.03177
```

Doc B wins because it ranked #1 in one retriever and #5 in the other -- consistent
presence across multiple retrievers is the signal RRF rewards.

## Implementing Hybrid Retrieve

```python
def rrf_fuse(ranked_lists, k=60):
    """
    ranked_lists: list of lists of doc-ids, each sorted best-first.
    Returns: dict {doc_id: rrf_score}, sorted descending.
    """
    scores = {}
    for ranked in ranked_lists:
        for rank, doc_id in enumerate(ranked):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank + 1)
    return dict(sorted(scores.items(), key=lambda x: x[1], reverse=True))

def hybrid_retrieve_demo(query, k=3):
    bm25_scores = bm25_index.get_scores(query.split())
    bm25_ranked = list(np.argsort(bm25_scores)[::-1])        # all chunks, ranked

    dense_docs   = faiss_index.similarity_search(query, k=len(CORPUS_TEXTS))
    dense_ranked = [CORPUS_TEXTS.index(d.page_content)        # map back to index
                    for d in dense_docs if d.page_content in CORPUS_TEXTS]

    fused = rrf_fuse([bm25_ranked, dense_ranked])
    top_k = list(fused.keys())[:k]
    return [(CORPUS_SOURCES[i], CORPUS_TEXTS[i][:120], round(fused[i], 5)) for i in top_k]
```

In [ ]:
# =============================================================================
# DEMO: RRF fusion combining BM25 + FAISS dense
# =============================================================================

def rrf_fuse(ranked_lists, k=60):
    """Combine multiple ranked lists using Reciprocal Rank Fusion.

    ranked_lists: list of lists of integer chunk indices, best-first.
    k=60: smoothing constant from Cormack et al. SIGIR 2009.
    Returns dict {chunk_idx: rrf_score} sorted descending.
    """
    scores = {}
    for ranked in ranked_lists:
        for rank, idx in enumerate(ranked):
            # 1-indexed rank inside the formula; rank=0 -> 1/(k+1)
            scores[idx] = scores.get(idx, 0.0) + 1.0 / (k + rank + 1)
    return dict(sorted(scores.items(), key=lambda x: x[1], reverse=True))


def hybrid_retrieve(query: str, k: int = 3,
                    bm25_idx=None, faiss_idx=None):
    """Hybrid BM25 + dense retrieval with RRF fusion.

    Both retrievers score ALL chunks; RRF re-ranks by position, not raw score.
    """
    bm25_idx   = bm25_idx   or bm25_index
    faiss_idx  = faiss_idx  or faiss_index

    # BM25: score all chunks, rank by descending score
    bm25_scores = bm25_idx.get_scores(query.split())
    bm25_ranked = list(np.argsort(bm25_scores)[::-1])

    # Dense: retrieve all chunks ordered by similarity
    n_chunks     = len(CORPUS_TEXTS)
    dense_docs   = faiss_idx.similarity_search(query, k=n_chunks)
    # Map back to corpus indices (exact text match)
    text_to_idx  = {t: i for i, t in enumerate(CORPUS_TEXTS)}
    dense_ranked = [text_to_idx[d.page_content]
                    for d in dense_docs if d.page_content in text_to_idx]

    fused = rrf_fuse([bm25_ranked, dense_ranked])
    top_k = list(fused.keys())[:k]
    return [(CORPUS_SOURCES[i], CORPUS_TEXTS[i][:120], round(fused[i], 6))
            for i in top_k]


# Run side-by-side comparison: BM25 vs dense vs hybrid
print("Retrieval comparison at k=3\n")
COMPARE_QUERIES = [
    "31 CFR 1010.311 currency transaction report",
    "what happens if a customer wires money late at night",
]
for q in COMPARE_QUERIES:
    b_res = bm25_retrieve(q, k=3)
    d_res = dense_retrieve(q, k=3)
    h_res = hybrid_retrieve(q, k=3)
    print(f"Q: {q!r}")
    print(f"  BM25:   {[s for s,_,_ in b_res]}")
    print(f"  Dense:  {[s for s,_,_ in d_res]}")
    print(f"  Hybrid: {[s for s,_,_ in h_res]}")
    print()

## Lab 2: Build the Hybrid Retriever

**Goal**: Implement `hybrid_retrieve_lab(query, k=3)` using the `rrf_fuse` helper
already defined above. Then run a hit-rate experiment across 6 queries to verify that
hybrid beats either retriever alone on at least 4/6.

**Steps**:

1. Inside `hybrid_retrieve_lab`, compute BM25 scores for all corpus chunks using
   `lab1_bm25.get_scores(query.split())` (from Lab 1). Rank descending.
2. Retrieve ALL chunks from `faiss_index.similarity_search(query, k=len(CORPUS_TEXTS))`.
   Map each result back to its corpus index via `CORPUS_TEXTS.index(d.page_content)`.
3. Pass both ranked lists to `rrf_fuse([bm25_ranked, dense_ranked])`.
4. Slice the top-k from the fused dict and return `(source, snippet, rrf_score)` tuples.
5. The evaluation loop below runs the comparison.

**Hint**: `np.argsort(scores)[::-1]` gives descending-order indices.

**Expected output**:
```
Query                                     BM25  Dense  Hybrid
31 CFR 1010.311 currency threshold         T      F       T
high risk MCC cryptocurrency               T      T       T
unusual hours 1 AM                         F      T       T
...
Hit rates:  BM25=4/6  Dense=4/6  Hybrid=6/6
```
(Exact numbers will vary; hybrid should match or beat both.)

In [ ]:
# Lab 2: Hybrid retriever with RRF
# -----------------------------------------------------------------------
def hybrid_retrieve_lab(query: str, k: int = 3):
    """Implement hybrid BM25 + dense retrieval fused with RRF."""
    # Step 1: BM25 scores and ranking (use lab1_bm25 from Lab 1)
    bm25_scores = None  # YOUR CODE
    bm25_ranked = None  # YOUR CODE  (descending order indices)

    # Step 2: Dense retrieval for all chunks
    n_chunks     = len(CORPUS_TEXTS)
    dense_docs   = None  # YOUR CODE  (similarity_search with k=n_chunks)
    text_to_idx  = {t: i for i, t in enumerate(CORPUS_TEXTS)}
    dense_ranked = None  # YOUR CODE  (list of corpus indices)

    # Step 3: RRF fusion
    fused = None  # YOUR CODE  (call rrf_fuse)

    # Step 4: return top-k (source, snippet, rrf_score)
    top_k = list(fused.keys())[:k]
    return [(CORPUS_SOURCES[i], CORPUS_TEXTS[i][:120], round(fused[i], 6))
            for i in top_k]


# -----------------------------------------------------------------------
# Evaluation loop (do not modify)
LAB2_QUERIES = [
    ("31 CFR 1010.311 currency transaction",       "ctr_rules.md"),
    ("high risk MCC cryptocurrency exchange",      "high_risk_merchant_categories.md"),
    ("unusual hours 1 AM 5 AM transactions",       "unusual_hours_rule.md"),
    ("what indicates suspicious wire transfer",    "account_takeover_patterns.md"),
    ("new payee hold 72 hours verification",       "new_payee_hold.md"),
    ("OFAC SDN sanctions screening",               "ofac_screening.md"),
]

if hybrid_retrieve_lab is not None and lab1_bm25 is not None:
    print(f"{'Query':<44} {'BM25':>5} {'Dense':>6} {'Hybrid':>7}")
    print("-" * 66)
    b_hits = d_hits = h_hits = 0
    for q, expected in LAB2_QUERIES:
        b = any(s == expected for s,_,_ in bm25_retrieve_lab(q, k=3))
        d = any(s == expected for s,_,_ in dense_retrieve(q,   k=3))
        h = any(s == expected for s,_,_ in hybrid_retrieve_lab(q, k=3))
        b_hits += b; d_hits += d; h_hits += h
        print(f"  {q[:42]:<42} {str(b):>5} {str(d):>6} {str(h):>7}")
    n = len(LAB2_QUERIES)
    print(f"\nHit rates @ k=3:  BM25={b_hits}/{n}  Dense={d_hits}/{n}  Hybrid={h_hits}/{n}")

# Section 3: Anthropic Contextual Retrieval

## The Problem: Chunks Lose Their Context

When you split a document into chunks, each chunk is embedded and stored in isolation.
Consider this chunk from `wire_record_keeping.md`:

> "Records must be retained for five years."

Without context, this chunk could match ANY query about record retention -- HIPAA,
tax records, anything. The retriever has no idea this sentence is specifically about
**wire transfer records under 31 CFR 1010.410**.

**Contextual Retrieval** (Anthropic, Nov 2024) fixes this by prepending a short
LLM-generated context sentence to each chunk BEFORE indexing. The context anchors
the chunk to its parent document:

> "This chunk is from wire_record_keeping.md which covers wire transfer recordkeeping
> under 31 CFR 1010.410. Records must be retained for five years."

Both the BM25 tokenized list AND the FAISS embedding are built from this contextual
version. This has been shown to reduce retrieval failure by 49% vs. standard hybrid
search (source: Anthropic contextual retrieval blog, Nov 2024).

## The Context Generation Pattern

```python
CONTEXT_PROMPT = (
    "Document: {doc_name}\n\n"
    "Chunk: {chunk_text}\n\n"
    "Write one sentence (max 30 words) that situates this chunk within its parent "
    "document. Start with 'This chunk is from {doc_name} which covers...'."
)

def generate_context(doc_name: str, chunk_text: str) -> str:
    resp = langchain_llm.invoke(
        CONTEXT_PROMPT.format(doc_name=doc_name, chunk_text=chunk_text[:400])
    )
    return resp.content.strip()
```

After generating context for every chunk, the contextual text is:
```
context_sentence + " " + original_chunk_text
```

This becomes the new chunk text for both BM25 and FAISS indexing.

In [ ]:
# =============================================================================
# DEMO: Context generation for a sample of chunks
# =============================================================================

CONTEXT_PROMPT = (
    "Document: {doc_name}\n\n"
    "Chunk: {chunk_text}\n\n"
    "Write one sentence (max 30 words) that situates this chunk within its parent "
    "document. Start with: This chunk is from {doc_name} which covers"
)

def generate_context(doc_name: str, chunk_text: str) -> str:
    """Call Haiku 3 to generate a context prefix for a chunk."""
    prompt = CONTEXT_PROMPT.format(
        doc_name=doc_name,
        chunk_text=chunk_text[:400]
    )
    resp = langchain_llm.invoke(prompt)
    return resp.content.strip()


# Demo: generate context for 3 representative chunks
DEMO_CHUNKS = [
    ("wire_record_keeping.md",
     "Records must be retained for five years."),
    ("ctr_rules.md",
     "Structuring - breaking a transaction into smaller amounts to evade the "
     "CTR threshold - is itself a federal violation under 31 USC 5324."),
    ("unusual_hours_rule.md",
     "Two or more such transactions within a single session should trigger "
     "a SAR review."),
]

print("Context generation demo (3 chunks)\n")
for doc_name, chunk_text in DEMO_CHUNKS:
    ctx = generate_context(doc_name, chunk_text)
    contextual_text = ctx + " " + chunk_text
    print(f"  [{doc_name}]")
    print(f"  Original : {chunk_text[:80]}...")
    print(f"  Context  : {ctx}")
    print(f"  Prepended: {contextual_text[:120]}...")
    print()

In [ ]:
# =============================================================================
# DEMO: Build contextual BM25 + FAISS indexes over the full corpus
# =============================================================================
# This takes ~30 seconds (one Haiku 3 call per chunk, 8 chunks total).

print(f"Building contextual indexes for {len(chunks)} chunks...")
print("Calling Haiku 3 to generate context for each chunk...\n")

contextual_texts   = []
contextual_sources = []

for i, chunk in enumerate(chunks):
    doc_name   = chunk.metadata["source"]
    chunk_text = chunk.page_content
    ctx        = generate_context(doc_name, chunk_text)
    full_text  = ctx + " " + chunk_text
    contextual_texts.append(full_text)
    contextual_sources.append(doc_name)
    print(f"  [{i+1}/{len(chunks)}] {doc_name}: {ctx[:70]}...")

# BM25 contextual index
ctx_tokenized = [t.split() for t in contextual_texts]
ctx_bm25      = BM25Okapi(ctx_tokenized)

# FAISS contextual index
ctx_documents  = [Document(page_content=t, metadata={"source": s})
                  for t, s in zip(contextual_texts, contextual_sources)]
ctx_faiss      = FAISS.from_documents(ctx_documents, bedrock_embeddings)

print(f"\nContextual indexes built: {len(contextual_texts)} chunks")

## Lab 3: Build the Contextual Hybrid Retriever

**Goal**: Implement `hybrid_contextual_retrieve(query, k=3)` that uses the
contextual BM25 (`ctx_bm25`) and contextual FAISS (`ctx_faiss`) indexes you
just built. This function will be used in the RAGAS evaluation in Section 4.

**Steps**:

1. Score all contextual chunks with `ctx_bm25.get_scores(query.split())`.
   Sort descending to get `ctx_bm25_ranked`.
2. Retrieve all chunks from `ctx_faiss.similarity_search(query, k=len(contextual_texts))`.
   Build `ctx_dense_ranked` by mapping page content back to corpus index using
   `{t: i for i, t in enumerate(contextual_texts)}`.
3. Call `rrf_fuse([ctx_bm25_ranked, ctx_dense_ranked])` to get the fused scores.
4. Slice the top-k and return `(source, contextual_text[:120], rrf_score)` tuples.
   Use `contextual_sources` and `contextual_texts` (not the original `CORPUS_SOURCES`
   and `CORPUS_TEXTS`) for the returned metadata.

**Expected output**: A 3-row table for each test query showing source filename and
the first 120 chars of the contextual chunk (should start with "This chunk is from...").

**Why this matters**: In Section 4 you will compare hit rates of:
- Standard hybrid (Section 2) using original chunks
- Contextual hybrid (this lab) using context-prepended chunks

The contextual version should have a higher RAGAS answer_relevancy score.

In [ ]:
# Lab 3: Contextual hybrid retriever
# -----------------------------------------------------------------------
def hybrid_contextual_retrieve(query: str, k: int = 3):
    """Hybrid retrieval over contextual (context-prepended) indexes."""
    # Step 1: BM25 over contextual chunks
    ctx_bm25_scores = None  # YOUR CODE  (use ctx_bm25)
    ctx_bm25_ranked = None  # YOUR CODE  (descending by score)

    # Step 2: Dense retrieval over contextual FAISS index
    ctx_dense_docs   = None  # YOUR CODE  (use ctx_faiss, k=len(contextual_texts))
    ctx_text_to_idx  = {t: i for i, t in enumerate(contextual_texts)}
    ctx_dense_ranked = None  # YOUR CODE  (list of corpus indices)

    # Step 3: RRF fusion
    ctx_fused = None  # YOUR CODE

    # Step 4: return top-k
    top_k = list(ctx_fused.keys())[:k]
    return [(contextual_sources[i], contextual_texts[i][:120], round(ctx_fused[i], 6))
            for i in top_k]


# -----------------------------------------------------------------------
# Sanity check: run 2 queries and inspect returned chunks
SANITY_QUERIES = [
    "31 CFR 1010.311 currency transaction",
    "what happens if a customer wires money late at night",
]

if hybrid_contextual_retrieve is not None:
    for q in SANITY_QUERIES:
        print(f"Query: {q!r}")
        try:
            results = hybrid_contextual_retrieve(q, k=3)
            for src, snippet, score in results:
                print(f"  [{src}] rrf={score}  '{snippet}...'")
        except Exception as e:
            print(f"  Error (finish the implementation): {e}")
        print()

In [ ]:
# Lab 3 safety-net: run this cell ONLY if you did not finish Lab 3.
# Skip this cell if hybrid_contextual_retrieve is already working above.
# SAFETY-NET
try:
    _test = hybrid_contextual_retrieve("test query", k=1)
    if _test is None or len(_test) == 0:
        raise ValueError("empty result")
except Exception:
    print("Using Lab 3 safety-net so Section 4 can run.")

    def hybrid_contextual_retrieve(query: str, k: int = 3):
        """Contextual hybrid retriever (safety-net implementation)."""
        ctx_bm25_scores = ctx_bm25.get_scores(query.split())
        ctx_bm25_ranked = list(np.argsort(ctx_bm25_scores)[::-1])

        n_ctx            = len(contextual_texts)
        ctx_dense_docs   = ctx_faiss.similarity_search(query, k=n_ctx)
        ctx_text_to_idx  = {t: i for i, t in enumerate(contextual_texts)}
        ctx_dense_ranked = [ctx_text_to_idx[d.page_content]
                            for d in ctx_dense_docs if d.page_content in ctx_text_to_idx]

        ctx_fused = rrf_fuse([ctx_bm25_ranked, ctx_dense_ranked])
        top_k     = list(ctx_fused.keys())[:k]
        return [(contextual_sources[i], contextual_texts[i][:120], round(ctx_fused[i], 6))
                for i in top_k]

    print("hybrid_contextual_retrieve is now defined via safety-net.")

# Section 4: RAGAS Evaluation -- Hybrid vs Contextual Hybrid

## Measuring What Actually Matters

Hit rate tells you whether the right document was retrieved. But you also need to
know whether the answers generated FROM those retrieved chunks are any good.

We use two RAGAS metrics:

| Metric | What it measures | Scale |
|--------|-----------------|-------|
| `faithfulness` | Is the answer grounded in the retrieved context? (no hallucination) | 0-1, higher = better |
| `answer_relevancy` | How relevant is the answer to the question? | 0-1, higher = better |

We compare three retrieval strategies:

1. **Dense-only**: FAISS cosine similarity (Week 17 baseline)
2. **Hybrid**: BM25 + FAISS + RRF (Section 2, standard chunks)
3. **Contextual Hybrid**: BM25 + FAISS + RRF on context-prepended chunks (Section 3)

The evaluation uses 4 representative fraud compliance questions. For each strategy,
we retrieve 3 chunks, feed them to Haiku 3, and score the answer with RAGAS.

In [ ]:
# =============================================================================
# DEMO: RAGAS evaluation comparing dense vs hybrid vs contextual hybrid
# =============================================================================

EVAL_QUESTIONS = [
    "What is the dollar threshold that triggers a Currency Transaction Report?",
    "What are the high-risk merchant category codes for cryptocurrency?",
    "When does the unusual hours rule require enhanced monitoring?",
    "What are the record retention requirements for international wire transfers?",
]

def build_answer(query: str, retrieved_chunks: list) -> str:
    """Generate an answer from retrieved chunks using Haiku 3."""
    context = "\n\n".join(text for _, text, _ in retrieved_chunks)
    prompt  = (
        f"Answer the question using only the context below.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n\nAnswer:"
    )
    return langchain_llm.invoke(prompt).content.strip()


def run_ragas_eval(retriever_fn, strategy_name: str):
    """Run RAGAS faithfulness + answer_relevancy for a retriever function."""
    samples = []
    for q in EVAL_QUESTIONS:
        chunks  = retriever_fn(q, k=3)
        answer  = build_answer(q, chunks)
        context = [text for _, text, _ in chunks]
        samples.append(SingleTurnSample(
            user_input=q,
            response=answer,
            retrieved_contexts=context,
        ))
    dataset = EvaluationDataset(samples=samples)
    results = evaluate(
        dataset=dataset,
        metrics=[faithfulness, answer_relevancy],
        llm=evaluator_llm,
        embeddings=evaluator_embeddings,
    )
    df = results.to_pandas()
    mean_faith = df["faithfulness"].mean()
    mean_relev = df["answer_relevancy"].mean()
    print(f"  {strategy_name:<25} faithfulness={mean_faith:.3f}  answer_relevancy={mean_relev:.3f}")
    return mean_faith, mean_relev


print("RAGAS evaluation (4 questions x 3 strategies)\n")
print(f"  {'Strategy':<25} {'Faithfulness':>12}  {'Ans Relevancy':>13}")
print("-" * 55)

dense_f,  dense_r  = run_ragas_eval(dense_retrieve,              "Dense-only")
hybrid_f, hybrid_r = run_ragas_eval(hybrid_retrieve,             "Hybrid (RRF)")
ctx_f,    ctx_r    = run_ragas_eval(hybrid_contextual_retrieve,  "Contextual Hybrid")

print("\n")
print(f"  Contextual hybrid faithfulness improvement:     {(ctx_f  - dense_f)*100:+.1f}%")
print(f"  Contextual hybrid answer relevancy improvement: {(ctx_r  - dense_r)*100:+.1f}%")

## Lab 4: Your Own RAGAS Comparison

**Goal**: Add two more evaluation questions to the mix and re-run the RAGAS evaluation
across all three strategies. Observe whether the contextual hybrid advantage holds on
your new questions.

**Steps**:

1. Add two new fraud compliance questions to `LAB4_QUESTIONS` below.
   Choose questions that are likely to benefit from hybrid search (e.g., questions
   referencing specific regulation codes, thresholds, or MCC codes).
2. Run the evaluation loop (provided). It calls `run_ragas_eval` on all three
   strategies with your combined 6-question set (`EVAL_QUESTIONS + LAB4_QUESTIONS`).
3. Print a comparison table and a one-sentence conclusion.

**Good question examples**:
- "Under 31 CFR 1010.410, how long must wire transfer records be kept?"
- "What is the wire transfer threshold for requiring additional due diligence?"
- "What must happen within 24 hours for OFAC false positives?"

**Expected output**:
```
Dense-only             faithfulness=0.XX  answer_relevancy=0.XX
Hybrid (RRF)           faithfulness=0.XX  answer_relevancy=0.XX
Contextual Hybrid      faithfulness=0.XX  answer_relevancy=0.XX
```

In [ ]:
# Lab 4: Extended RAGAS comparison with your own questions
# -----------------------------------------------------------------------
# Step 1: Add two new questions (pick ones with specific terms/codes)
LAB4_QUESTIONS = [
    None,  # YOUR CODE - replace with a fraud compliance question (string)
    None,  # YOUR CODE - replace with a second question (string)
]

# -----------------------------------------------------------------------
# Evaluation loop (do not modify below)
all_questions = EVAL_QUESTIONS + [q for q in LAB4_QUESTIONS if q is not None]

print(f"Running RAGAS on {len(all_questions)} questions x 3 strategies\n")
print(f"  {'Strategy':<25} {'Faithfulness':>12}  {'Ans Relevancy':>13}")
print("-" * 55)

def run_ragas_eval_on(questions, retriever_fn, strategy_name):
    samples = []
    for q in questions:
        chunks  = retriever_fn(q, k=3)
        answer  = build_answer(q, chunks)
        context = [text for _, text, _ in chunks]
        samples.append(SingleTurnSample(
            user_input=q,
            response=answer,
            retrieved_contexts=context,
        ))
    dataset = EvaluationDataset(samples=samples)
    results = evaluate(
        dataset=dataset,
        metrics=[faithfulness, answer_relevancy],
        llm=evaluator_llm,
        embeddings=evaluator_embeddings,
    )
    df = results.to_pandas()
    mf = df["faithfulness"].mean()
    mr = df["answer_relevancy"].mean()
    print(f"  {strategy_name:<25} faithfulness={mf:.3f}  answer_relevancy={mr:.3f}")
    return mf, mr

df_f, df_r = run_ragas_eval_on(all_questions, dense_retrieve,             "Dense-only")
hf_f, hf_r = run_ragas_eval_on(all_questions, hybrid_retrieve,            "Hybrid (RRF)")
cf_f, cf_r = run_ragas_eval_on(all_questions, hybrid_contextual_retrieve, "Contextual Hybrid")

print(f"\nContextual vs Dense: faithfulness {(cf_f-df_f)*100:+.1f}%  relevancy {(cf_r-df_r)*100:+.1f}%")

# Summary: What You Built

Congratulations -- you have implemented production-grade hybrid retrieval from scratch.

## What You Learned

| Concept | Key Insight |
|---------|-------------|
| BM25 | Keyword-based retrieval wins on exact terms (regulation codes, thresholds, MCCs) |
| Dense | Semantic retrieval wins on paraphrased, conceptual queries |
| RRF | Rank-based fusion sidesteps score incompatibility; k=60 is the standard constant |
| Contextual Retrieval | LLM-generated context prefixes reduce retrieval failure by ~49% (Anthropic, 2024) |
| RAGAS | Faithfulness + answer_relevancy measure end-to-end RAG quality, not just hit rate |

## The Production Pattern

```
Query
  |
  +-- BM25 (sparse, exact-term)     --\
  |                                    > RRF (k=60)  --> Top-k chunks --> LLM --> Answer
  +-- FAISS dense (semantic)        --/
       |
       Both indexed on: context_prefix + original_chunk
```

This is the retrieval architecture used by Elastic, Pinecone, Amazon Bedrock,
and Anthropic's own reference RAG implementations.

## Key Takeaways

- Do not combine raw BM25 and cosine scores -- they are incomparable. Use RRF.
- Tokenize BM25 with `.split()`, not subword tokenizers. Exact tokens matter.
- Contextual retrieval is a simple 1-API-call-per-chunk investment with large payoff.
- RAGAS faithfulness and answer_relevancy complement hit rate -- you need both.

## Resources

- Cormack et al., "Reciprocal Rank Fusion outperforms Condorcet and individual
  Rank Learning Methods" -- SIGIR 2009 (source of k=60)
- Anthropic Contextual Retrieval blog -- Nov 2024
- rank_bm25 docs: https://github.com/dorianbrown/rank_bm25
- RAGAS docs: https://docs.ragas.io